In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.ndimage import gaussian_filter
import pandas as pd
import scipy.ndimage as sp
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from cycler import cycler
import scipy.stats as stats
from tqdm import tqdm

import cv2
from skimage import measure
import os

from skimage import io, exposure, data
from skimage import feature


mpl.rcParams.update(mpl.rcParamsDefault)
mpl.rcParams['pdf.fonttype'] = 42

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

pd.set_option('display.float_format', '{:.20f}'.format)

In [ ]:
#---------------------change directory folder name----------------



folder_name = ''
directory_name = ''


#-----------------------define functions--------------------------



def string_to_float_array(string):
    return [float(x) for x in string.split(',')]


def isolate_puncta(input_image_path, output_mask_path, output_expanded_path, output_coordinates_path, threshold_value):
    # Load the image
    image = cv2.imread(input_image_path, cv2.IMREAD_GRAYSCALE)
    
    # Apply a threshold to separate puncta from the background
    _, binary_mask = cv2.threshold(image, threshold_value, 255, cv2.THRESH_BINARY)
   
    # Expand the isolated mask by 3 pixels
    isolated_puncta = np.zeros_like(image)
    kernel = np.ones((3, 3), np.uint8)
    expanded_mask = cv2.dilate(binary_mask, kernel, iterations=1)
    
    def get_mask_coordinates(mask):

    # Get the coordinates where the mask is 1
        coordinates = np.argwhere(mask != 0)
        return coordinates
    coordinates = get_mask_coordinates(expanded_mask)
    
    
    # Save the binary mask, isolated puncta, and pixel coordinates
    cv2.imwrite(output_mask_path, binary_mask)
    cv2.imwrite(output_isolated_path, isolated_puncta)
    cv2.imwrite(output_expanded_path, expanded_mask)
    
    # Create a DataFrame from the pixel coordinates
    df_coordinates = pd.DataFrame(coordinates, columns=["Y", "X"])
    #df_coordinates = pd.DataFrame(coordinates, columns=["X", "Y"])
    
    # Save the pixel coordinates as a CSV file
    df_coordinates.to_csv(output_coordinates_path, index=False)


def row_differences(group):
    # Compute differences within each group
    group['distance_sq'] = group['X'].diff(periods=1)**2 + group['Y'].diff(periods=1)**2
    group['time_int'] = group['time'].diff(periods=1)
    return group

def row_single_differences(group):
    # Compute differences within each group
    group['X_diff'] = group['X'].diff(periods=1)
    group['Y_diff'] = group['Y'].diff(periods=1)
    return group




#----------MINFLUX tracks NP array to Pandas-----------------



min_total_total = pd.DataFrame()


for i in range(1,2):

    if i <10:
        min_filename = folder_name + ' 0' + str(i)
    else:
        min_filename = folder_name + ' ' + str(i)
    directory = directory_name + folder_name + '/' + min_filename + '/'
    minflux = np.load(directory + min_filename + '.npy')

    

    
    

    #---for new format--------
    itr3 = np.extract(minflux['itr'] ==3, minflux)
    minf = pd.DataFrame({'track_id': itr3['tid'], 'time' : itr3['tim'], 'X':itr3['loc'][:,0], 'Y':itr3['loc'][:,1], 'status':itr3['sta'], 'itr': itr3['itr'], 'vld': itr3['vld'], 'final':itr3['fnl'], 'bot': itr3['bot'], 'end': itr3['eot'] })
    min_data_s = minf.sort_values(by=['track_id', 'time'])
    min_data_s = min_data_s[min_data_s['itr']==3]

    
    #---for old format---------
    #pos = pd.DataFrame((minflux['itr'][:,3]['loc']), columns = ['X','Y','Z'])
    #minf = pd.DataFrame({'track_id': minflux['tid'], 'time' : minflux['tim'], 'vld': minflux['vld'] })
    #min_data = pd.concat([minf, pos], axis=1)
    #min_data_s = min_data.sort_values(by=['track_id', 'time']) 
    
    
    
    

    

#--------------------coordinate conversion- Lengths and offsets---------------
    
    if os.path.exists(directory + '/params.csv'):
        param = pd.read_csv(directory + '/params.csv')
        
        L_str = param['0'][0:1][0]
        L = string_to_float_array(L_str[L_str.find('[')+1 : L_str.find(']')])

        
        O_str = "".join(param['0'][1:2][1])
        O = string_to_float_array(O_str[O_str.find('[')+1 : O_str.find(']')])

    else:
        metadata = pd.read_csv(directory +  'Original Metadata - ' + min_filename + '.csv' ,sep='\t')

        L_str = metadata[25:26]['Key,Value'].astype(str).iloc[0]
        L_str = L_str[L_str.find('[')+1 : L_str.find(']')]
        L = string_to_float_array(L_str)

        O_str = metadata[27:28]['Key,Value'].astype(str).iloc[0]
        O_str = O_str[O_str.find('[')+1 : O_str.find(']')]
        O = string_to_float_array(O_str)

    Lengths = [i * 1e6 for i in L]
    Offsets = [i * 1e6 for i in O]

    min_data_s.X = (min_data_s.X*1e6)-Offsets[0]
    min_data_s.Y = (min_data_s.Y*1e6)-Offsets[1]

    param = pd.DataFrame({'L': [L], 'O':[O], 'filename':min_filename})
    param = param.T

    #min_data_s.to_csv(directory + '/spots.csv')
    param.to_csv(directory + '/params.csv')
    
    
    
    #filter tracks based on length and displacement to get rid of aggregates/noise
    
    min_data_s = min_data_s.groupby('track_id').filter(lambda x: 
                                                       (x['time'].count() > 500) & 
                                                       (x['X'].max() - x['X'].min() > 0.2) & 
                                                       (x['Y'].max() - x['Y'].min() > 0.2) & 
                                                       (((x['X'].iloc[-1] - x['X'].iloc[0])**2) + 
                                                        ((x['Y'].iloc[-1] - x['Y'].iloc[0])**2) > 0.01))
     
    #for some reason 1st loc of each track is somewhat off, so removing the 
    #very first element without touching the rest could solve the issue
    min_data_s = min_data_s.groupby(['track_id']).apply(lambda x: x.iloc[1:], include_groups=False).reset_index()

#-----------------------LD confocal image-----------------------------


    cmap3 = LinearSegmentedColormap.from_list('mycmap', ['white', 'darkgreen'])
    cmap_w = LinearSegmentedColormap.from_list('mycmap', ['white', 'white'])
    min_LDs = directory + 'LDs.tif'

    IMAGE = io.imread(min_LDs)
    pre = IMAGE[0]
    post = IMAGE[1]
    MIP= np.max(IMAGE, axis=0)
    image_minmax_scaled = exposure.rescale_intensity(MIP)
    percentiles = np.percentile(image_minmax_scaled, (40, 99))
    scaled = exposure.rescale_intensity(MIP,
                                        in_range=tuple(percentiles))
    pre_LDs = exposure.rescale_intensity(pre,
                                        in_range=tuple(percentiles))
    post_LDs = exposure.rescale_intensity(post,
                                        in_range=tuple(percentiles))

    
    #for plotting - purely visual
    dot_size = 8000/MIP.size

 
    
#-------------------------masking and track categorization--------------------------
    
    
    threshold_value = 25
    pixel_size = 70
    
    ratio = (1000/pixel_size)
    
    coor = pd.DataFrame({'track_id': min_data_s.track_id, 'X': min_data_s.X*ratio, 'Y': min_data_s.Y*ratio, 'time':min_data_s.time}, columns=['track_id','X', 'Y', 'time'])      

    
    if __name__ == "__main__":
        # Set your input and output paths
        directory_mask = directory
        input_image_path = directory_mask + "MAX_LDs.tif"
        output_mask_path = directory_mask +"mask.png"
        output_isolated_path = directory_mask +"isolated_puncta.png"
        output_expanded_path = directory_mask +"expanded_puncta.png"
        output_coordinates_path = directory_mask +"coordinates.csv"

        
        threshold_value = threshold_value

        # Call the function to isolate puncta and collect coordinates
        isolate_puncta(input_image_path, output_mask_path, output_isolated_path, output_coordinates_path, threshold_value)
        mask_coordinates = pd.read_csv(output_coordinates_path, low_memory=False)

        inside = pd.DataFrame()
        for key,grp in mask_coordinates.groupby('X'):
            coord = coor[((coor['Y']//1).isin(grp.Y)) & ((coor['X']//1)==key) ]
            inside = pd.concat([inside, coord])

        outside = pd.concat([coor,inside]).drop_duplicates(keep=False)
        
        
        inside.X = inside.X/ratio
        inside.Y = inside.Y/ratio
        outside.X = outside.X/ratio
        outside.Y = outside.Y/ratio
        
        
        #inside.X = inside.X/20
        #inside.Y = inside.Y/20
        #outside.X = outside.X/20
        #outside.Y = outside.Y/20
        cmap_k = LinearSegmentedColormap.from_list('mycmap', ['white', 'black'])


    
    inside['Loc'] = 'inside'
    outside['Loc'] = 'outside'
    MIN_tracks = pd.concat([outside, inside], ignore_index=True).sort_values(by=['track_id', 'time'])
    
    
    
    
#------------------squared displacement---------------------------------

    
    MIN_track_info = MIN_tracks.groupby('track_id').apply(row_differences, include_groups=False).reset_index().drop(columns=['level_1'])
    
    #additional filtering
    edge_tracks = MIN_track_info[(MIN_track_info['X']<=0) & (MIN_track_info['Y']<=0)]
    MIN_track_info = MIN_track_info[~MIN_track_info['track_id'].isin(edge_tracks)]
    
    MIN_track_info['steps'] = np.sqrt(MIN_track_info['distance_sq'])
    MIN_track_info['filename']=min_filename

    

#---------------------------set single vs multiple entry----------------------

    location_sets =  MIN_track_info.groupby(['filename', 'track_id']).Loc.apply(set, include_groups=False).reset_index()
    inside = MIN_track_info[MIN_track_info['track_id'].isin(location_sets[location_sets['Loc']=={'inside'}].track_id)].copy()
    inside['type'] = 'single'

    outside = MIN_track_info[MIN_track_info['track_id'].isin(location_sets[location_sets['Loc']=={'outside'}].track_id)].copy()
    outside['type'] = 'single'

    mixed = MIN_track_info[(~MIN_track_info['track_id'].isin(inside.track_id)) & (~MIN_track_info['track_id'].isin(outside.track_id))].copy()
    mixed['type'] = 'multiple switches'

 

    
#----------------------update multiple switches if most of the track is on LDs-------------------------

    # Function to update Loc based on the condition
    
    def update_loc(df):
        counts = df.groupby('track_id')['Loc'].value_counts().unstack(fill_value=0)

        for key, row in counts.iterrows():
            num_inside = row.get('inside', 0)
            num_outside = row.get('outside', 0)

            if num_outside >= 8 * num_inside:
                df.loc[(df['track_id'] == key), 'Loc'] = 'outside'
                df.loc[(df['track_id'] == key), 'type'] = 'single'
            elif num_inside >= 8 * num_outside:
                df.loc[(df['track_id'] == key), 'Loc'] = 'inside'
                df.loc[(df['track_id'] == key), 'type'] = 'single'

        return df

    updated_mixed = update_loc(mixed)
    min_total = pd.concat([inside, outside, updated_mixed])
    

     
    

#----------------------save LD image and tracks overlaid------------------------

    
    plt.imshow(scaled, cmap=cmap3, extent=[0,Lengths[0],Lengths[1],0])
    sns.scatterplot(data= min_total, x = 'X', y = 'Y', hue = 'track_id', s = dot_size, alpha = 0.85, linewidth = 0, 
                    legend = False, palette = "Set2", edgecolor=None)
    sns.lineplot(data= min_total, x = 'X', y = 'Y', hue = 'track_id', lw = dot_size/4, sort = False, estimator = None, 
                  palette = 'Set2', legend = False, alpha = 0.6)

    plt.xlim(0,Lengths[0])
    plt.ylim(0, Lengths[1])
    plt.xlabel ('X position (µm)')
    plt.ylabel ('Y position (µm)')
    plt.gca().invert_yaxis()
    plt.savefig(directory  +'Tracks_and_LDs.png', dpi = 2400)
    plt.clf()


    
#--------------------single tracks--------------------------------------
    
    
    plt.imshow(scaled, cmap=cmap3, extent=[0,Lengths[0],Lengths[1],0])
    sns.scatterplot(data= min_total[min_total['type']=='single'], x = 'X', y = 'Y', hue = 'track_id', s = dot_size, alpha = 0.85,  
                    legend = False, palette = "Set2", edgecolor=None)
    sns.lineplot(data= min_total[min_total['type']=='single'], x = 'X', y = 'Y', hue = 'track_id', lw = dot_size, sort = False, estimator = None, 
                  palette = 'Set2', legend = False, alpha = 0.6)

    plt.xlim(0,Lengths[0])
    plt.ylim(0, Lengths[1])
    plt.xlabel ('X position (µm)')
    plt.ylabel ('Y position (µm)')
    plt.title('single localization tracks')
    plt.gca().invert_yaxis()
    plt.savefig(directory  +'Tracks_and_LDs_singles.png', dpi = 2400)
    plt.clf()



#--------------------multiple switching tracks--------------------------

    plt.imshow(scaled, cmap=cmap3, extent=[0,Lengths[0],Lengths[1],0])
    sns.scatterplot(data= min_total[min_total['type']=='multiple switches'], x = 'X', y = 'Y', hue = 'track_id', s = dot_size, alpha = 0.85,  
                    legend = False, palette = 'Set2', edgecolor=None)
    sns.lineplot(data= min_total[min_total['type']=='multiple switches'], x = 'X', y = 'Y', hue = 'track_id', lw = dot_size, sort = False, estimator = None, 
                  palette = 'Set2', legend = False, alpha = 0.6)

    plt.xlim(0,Lengths[0])
    plt.ylim(0, Lengths[1])
    plt.xlabel ('X position (µm)')
    plt.ylabel ('Y position (µm)')
    plt.title('multiple localization tracks')
    plt.gca().invert_yaxis()
    plt.savefig(directory  +'Tracks_and_LDs_multiples.png', dpi = 2400)
    plt.clf()


    
#--------------------------plot tracks individually-------------------------

    folder_path = os.path.join(directory, 'Tracks_v2')
    os.makedirs(folder_path, exist_ok=True)
    
    data = min_total
    
    for key,grp in data.groupby('track_id'):
        norm = plt.Normalize(round(data[data['track_id']==key].time.min(),2), round(data[data['track_id']==key].time.max(),2))
        sm = plt.cm.ScalarMappable(cmap="magma", norm=norm)
       
    
    
        """""
        if data[data['track_id']==key].time.median() < data.time.median():
            plt.imshow(pre_LDs, cmap=cmap3, extent=[0,Lengths[0],Lengths[1],0])
        else:
            plt.imshow(post_LDs, cmap=cmap3, extent=[0,Lengths[0],Lengths[1],0])
        """""
        
        
        plt.imshow(scaled, cmap=cmap3, extent=[0,Lengths[0],Lengths[1],0])
        sns.lineplot(data[data['track_id']==key], x = 'X', y = 'Y', c = 'Grey', linewidth = 0.1, alpha = 0.7,
                    sort = False, estimator = None)
        ax = sns.scatterplot(data[data['track_id']==key], x = 'X', y = 'Y', hue = 'time', s = 2, alpha = 0.7, 
                     legend = False, palette = 'magma',  linewidth=0)

        ax.figure.colorbar(sm, ax=ax, label='time (sec)', 
                       ticks=[round(data[data['track_id']==key].time.min(),2), round(data[data['track_id']==key].time.max(),2)],
                      orientation = 'horizontal', shrink = 0.2,  aspect = 9)

        plt.xlim(data[data['track_id']==key].X.min()-0.3, data[data['track_id']==key].X.max()+0.3)
        plt.ylim(data[data['track_id']==key].Y.min()-0.3, data[data['track_id']==key].Y.max()+0.3)
        plt.xlabel('X position (µm)')
        plt.ylabel('Y position (µm)')
        plt.title('Track_' + str(key))
        plt.gca().invert_yaxis()
        
        if grp['type'].iloc[0]=='multiple switches':
            plt.savefig(folder_path + '/MS_Track_' + str(key) + '.pdf', dpi = 600)
            plt.clf()
        else:
            if grp['Loc'].iloc[0]=='inside':
                plt.savefig(folder_path + '/I_Track_' + str(key) + '.pdf', dpi = 600)
                plt.clf()
            else:
                plt.savefig(folder_path + '/O_Track_' + str(key) + '.pdf', dpi = 600)
                plt.clf()
        
    

    
#--------------------plotting tracks based on Loc--------------------------
    
    
    data = min_total.sort_values(by=['Loc'], ascending = False)
    
    plt.imshow(scaled, cmap=cmap3, extent=[0,Lengths[0],Lengths[1],0])
    sns.scatterplot(data= data, x = 'X', y = 'Y', hue = 'Loc', s = dot_size, alpha = 0.7, 
                 legend = False, palette = 'magma',  linewidth=0)
    plt.title('Threshold value: ' + str(threshold_value))
    plt.xlabel('X (µm)')
    plt.ylabel('Y (µm)')
    plt.xlim(0,Lengths[0])
    plt.ylim(Lengths[1], 0)

    plt.savefig(directory_mask + 'masked_spots_locations.pdf', dpi = 300)
    plt.show()
    plt.clf()

    
    
#--------------------------masked tracks---------------------------------


    mask = io.imread(directory + 'isolated_puncta.png')
    #plt.imshow(mask)
    plt.imshow(mask, cmap=cmap3, extent=[0,Lengths[0],Lengths[1],0])
    sns.scatterplot(data= data, x = 'X', y = 'Y', hue = 'Loc', s = dot_size, alpha = 0.7, 
                     legend = True, palette = 'magma',  linewidth=0)
    
    plt.xlim(0,Lengths[0])
    plt.ylim(Lengths[1], 0)
    plt.xlabel('X (µm)')
    plt.ylabel('Y (µm)')
    plt.savefig(directory + 'mask_locations.pdf', dpi = 1200)
    plt.clf()
    
    
    
#-------------------density of tracks over LDs-------------------------
    
    
    plt.imshow(scaled, cmap=cmap3, extent=[0,Lengths[0],Lengths[1],0])
    plt.hist2d(data.X, data.Y, bins=(MIP.shape[1]*2, MIP.shape[0]*2), cmap = 'magma', cmin = 10)
    plt.colorbar(label='# of localizations)')
    plt.clim(0, 50)
    
    
    plt.xlim(0,Lengths[0])
    plt.ylim(Lengths[1], 0)
    plt.savefig(directory  +'density_map.png', dpi = 2400)
    plt.clf()
       
    
#--------------------plot jump distance-------------------------------
    
    
    sns.histplot(data = data, x = 'steps' , hue = 'Loc', palette = 'magma', bins=70, element="step", stat = 'density', common_norm=False, fill = True, kde = False)
    print(' ')
    print('inside: ' + str(data[data['Loc'] == 'inside'].steps.mean()))
    print('outside: ' + str(data[data['Loc'] == 'outside'].steps.mean()))

    #t_test = stats.ttest_ind(min_total[min_total['Loc'] == 'outside'].steps, min_total[min_total['Loc'] == 'inside'].steps, nan_policy='omit')
    #mann_whit = stats.mannwhitneyu(min_total[min_total['Loc'] == 'outside'].steps, min_total[min_total['Loc'] == 'inside'].steps, nan_policy='omit')
    plt.title('std_inside: ' + str(np.std(data[data['Loc']=='inside'].steps))  + 
              '\nstd_outside: ' +  str(np.std(data[data['Loc']=='outside'].steps))) 
    
    plt.savefig(directory + 'jump_distance.pdf', dpi = 1200)
    plt.show() 
    plt.clf()
    
    
    
    
#------------------localization precision---------------------------  
    
    
    
    mind_diff = min_total.groupby(['track_id']).apply(row_single_differences, include_groups=False).reset_index()


    plt.hist([mind_diff.X_diff*1000, mind_diff.Y_diff*1000], color=['chocolate','skyblue'], alpha=0.9, bins = 40)
    plt.legend(['ΔX','ΔY'])
    plt.title('st. dev. (ΔX): ' + str(np.std(mind_diff.X_diff)*1000) + 
              'nm' + '\nst. dev. (ΔY): ' + str(np.std(mind_diff.Y_diff)*1000) + 'nm') 
    plt.xlim(-50,50)
    plt.xlabel('difference between consecutive localizations (nm)')
    plt.savefig(directory  +'resolution.pdf', dpi = 600)
    plt.show()
    plt.clf()
    
    
#-------------------time interval---------------------------------
    

    
    sns.histplot(min_total.dropna().time_int, bins = 1000, color = 'grey')
    plt.xlim(0,0.002)
    plt.title('median: ' + str(round(min_total.dropna().time_int.median()*1000000,1)) + ' µsec' + '\nmean: ' + str(round(min_total.dropna().time_int.mean()*1000000,1)) + ' µsec')
    plt.savefig(directory + '/time_intervals.pdf', dpi = 300)
    plt.show()
    plt.clf()
    
    
    
    

    min_total_total = pd.concat([min_total_total, min_total])

    
    
min_total_total = min_total_total.drop_duplicates().sort_values(by=['filename','track_id', 'time'])
min_total_total.to_csv(directory_name + folder_name + '/'+ 'combined_all_tracks.csv')


In [ ]:
#sliding window msd_and D - 5ms windows

from tqdm import tqdm
time_frame = 0.005

def msd_analysis(df, window_size=int(time_frame/(min_total_total.time_int.median()))):
    results = []
   
    for (filename, track_id), track in tqdm(df.groupby(['filename', 'track_id'])):
        track = track.sort_values(by='time').reset_index(drop=True)
        num_points = len(track)
        
        if num_points < window_size:
            continue
        
        for i in range(num_points - window_size + 1):
            window = track.iloc[i:i+window_size]
            time_lags = window['time'].values - window['time'].values[0]
            displacements = (window['X'].values - window['X'].values[0])**2 + \
                            (window['Y'].values - window['Y'].values[0])**2
            
            msd_values = [np.mean(displacements[:j+1]) for j in range(1, window_size)]
            time_lags = time_lags[1:]
            
            if len(msd_values) < 2:
                continue
            
            # Calculate Brownian diffusion coefficient D = MSD / (4 * t)
            D_values = [msd / (4 * t) if t > 0 else np.nan for msd, t in zip(msd_values, time_lags)]
            D_avg = np.nanmean(D_values) if len(D_values) > 0 else np.nan
            
            results.append([filename, track_id,  window['time'].values[0], D_avg])
    
    rolling_D = pd.DataFrame(results, columns=['filename', 'track_id',  'start_time', 'D'])
    return rolling_D

df = min_total_total#[min_total_total['type']=='single']
rolling_D = msd_analysis(df)

rolling_D = rolling_D.merge(
    min_total_total[['filename', 'track_id', 'Loc', 'time', 'type', 'X', 'Y']],
    left_on=['filename', 'track_id',  'start_time'],
    right_on=['filename', 'track_id',  'time'],
    how='left'
)

rolling_D.drop(columns='time', inplace=True)

rolling_D.to_csv(directory_name + folder_name + '/sliding_msd_' + str(time_frame*1000) + 'ms.csv')


In [ ]:
# Rolling Dapp plot
plt.rc('font', family='Helvetica', size=20)
plt.figure(figsize=(6, 6))


#data = data[data['type']=='single']
sns.histplot(data = data, x = 'D', hue = 'Loc' ,palette = 'inferno', bins=20, element="step", 
                     stat = 'density', common_norm=False, fill = True, log_scale = True)

plt.title(str(data.groupby('Loc').D.median().apply(lambda x: f"{x:.3f} µm²/s")))
plt.xlabel('D_app')

plt.axvline(x=data[data['Loc']=='inside'].D.median(), color='salmon', linestyle='--')
plt.axvline(x=data[data['Loc']=='outside'].D.median(), color='purple', linestyle='--')



print(np.std(data[data['Loc']=='inside'].D))
print(np.std(data[data['Loc']=='outside'].D))


sns.despine()
plt.xlabel('Dapp (µm²/s)')
plt.tight_layout()
plt.savefig(directory_name + '/D_app_sliding_window_' + str(time_frame*1000) + 'ms.pdf', dpi = 300)
plt.show()

In [ ]:
#alpha calculation for 100ms segments

from scipy.optimize import curve_fit

msd_data = msd_info.groupby(['filename', 'track_id', 'loc']).filter(lambda x: (x['msd'].count() >= 500)) 

D_alpha = pd.DataFrame()
folder_path = os.path.join(directory_name  , 'power_fit_all_500')
os.makedirs(folder_path, exist_ok=True)


def power_law(t, D, alpha):
    return 4* D * t**alpha

#for lag_t in [0.005, 0.01, 0.05,0.1,0.5,1,5]:
for lag_t in [5]:
    
    D_alpha = pd.DataFrame()
    alpha_cal = pd.DataFrame()
    
    alpha_cal = msd_data[(msd_data['lag_time']>0) & (msd_data['lag_time']<lag_t)].dropna()#

    for key,grp in alpha_cal.groupby(['filename','track_id', 'type', 'loc']):
        if len(grp)> 5:
            time = np.array(grp.lag_time)  # time data
            msd = np.array(grp.msd)  # MSD data


        popt, pcov = curve_fit(power_law, time, msd, maxfev=25000)
        D, alpha = popt
        fitted_msd = power_law(time, D, alpha)
        perr = np.sqrt(np.diag(pcov))
        relative_uncertainty = (perr / np.abs(popt)) * 100

        values = pd.DataFrame({'filename': key[0], 'track_id': key[1], 'D': D, 'alpha':alpha, 'type':key[2], 'Loc':key[3], 
                               'relative_uncertainty_D': relative_uncertainty[0], 
                               'relative_uncertainty_alpha': relative_uncertainty[1]
                              }, 
                              columns=['filename','track_id', 'D', 'alpha', 'type', 'Loc', 
                                       'relative_uncertainty_D', 'relative_uncertainty_alpha'], 
                              index=[0])

        D_alpha = pd.concat([D_alpha, values])


    D_alpha_ = D_alpha[(D_alpha['alpha']<5) & (D_alpha['alpha']>0) & (D_alpha['D']<5) & ((D_alpha['relative_uncertainty_D']<25) |
                       (D_alpha['relative_uncertainty_alpha']<25))].sort_values(by=['filename', 'track_id', 'Loc'])



    values = pd.DataFrame({'param': ['lag_t','D_inside_mean', 'D_outside_mean', 'D_inside_median', 'D_outside_median', 'D_inside_std', 'D_outside_std',
                                    'alpha_inside_mean', 'alpha_outside_mean', 'alpha_inside_median', 'alpha_outside_median', 'alpha_inside_std', 
                                       'alpha_outside_std', '# of tracks', '# of inside LDs', '# of outside LDs', 'discarded'], 
                           'value': [lag_t, D_alpha_[D_alpha_['Loc']=='inside'].D.mean(),
                                     D_alpha_[D_alpha_['Loc']=='outside'].D.mean(),
                                     D_alpha_[D_alpha_['Loc']=='inside'].D.median(),
                                     D_alpha_[D_alpha_['Loc']=='outside'].D.median(),
                                    np.std(D_alpha_[D_alpha_['Loc']=='inside'].D),
                                    np.std(D_alpha_[D_alpha_['Loc']=='outside'].D),
                                    D_alpha_[D_alpha_['Loc']=='inside'].alpha.mean(),
                                     D_alpha_[D_alpha_['Loc']=='outside'].alpha.mean(),
                                     D_alpha_[D_alpha_['Loc']=='inside'].alpha.median(),
                                     D_alpha_[D_alpha_['Loc']=='outside'].alpha.median(),
                                    np.std(D_alpha_[D_alpha_['Loc']=='inside'].alpha),
                                    np.std(D_alpha_[D_alpha_['Loc']=='outside'].alpha),
                                    len(D_alpha_), 
                                    len(D_alpha_[D_alpha_['Loc']=='inside']),
                                    len(D_alpha_[D_alpha_['Loc']=='outside']),
                                    msd_data.groupby(['filename','loc'])['track_id'].nunique().sum()-len(D_alpha_)]})

    #values.to_csv(folder_path + '/mean_D_alpha_values_' + str(lag_t) + 'sec.csv')
    #D_alpha_.to_csv(folder_path + '/D_&_alpha_values_'  + str(lag_t) + 'sec.csv')

    display(values)
    
    plt.rc('font', family='Helvetica', size=20)
    plt.figure(figsize=(6, 6))
    
    if D_alpha_.Loc.iloc[0]=='inside':
        
        sns.histplot(data = D_alpha_, x = 'D', log_scale = True, palette = 'magma_r', hue = 'Loc', bins=20, element="step", 
                     stat = 'density', common_norm=False, fill = True)
        
        plt.xlabel('Dapp')
        plt.title('Dapp - ' + str(lag_t) + ' sec')
        plt.tight_layout()
        #plt.savefig(folder_path +'/D_'  + str(lag_t) + 'sec_power_law.pdf', dpi = 300)
        plt.show()
        plt.clf()


        sns.histplot(data = D_alpha_, x = 'alpha', log_scale = True, palette = 'magma_r', hue = 'Loc', bins=20, 
                     element="step", stat = 'density', common_norm=False, fill = True, kde = True, kde_kws={'bw_adjust': 2})
        
        plt.xlabel('alpha')
        plt.title('Alpha - ' + str(lag_t) + ' sec')
        plt.axvline(x=D_alpha_[D_alpha_['Loc']=='inside'].alpha.median(), color='salmon', linestyle='--')
        plt.axvline(x=D_alpha_[D_alpha_['Loc']=='outside'].alpha.median(), color='purple', linestyle='--')
        plt.tight_layout()
        #plt.savefig(folder_path + '/alpha_' + str(lag_t) +  'sec_power_law.pdf', dpi = 300)
        plt.show()
    
    else:
        sns.histplot(data = D_alpha_, x = 'D', log_scale = True, palette = 'magma', hue = 'Loc', bins=20, element="step", 
                     stat = 'density', common_norm=False, fill = True)
        plt.xlabel('Dapp')
        plt.title('Dapp - ' + str(lag_t) + ' sec')
        plt.tight_layout()
        #plt.savefig(folder_path +'/D_'  + str(lag_t) + 'sec_power_law.pdf', dpi = 300)
        plt.show()
        plt.clf()


        sns.histplot(data = D_alpha_, x = 'alpha', log_scale = True, palette = 'magma', hue = 'Loc', bins=20, 
                     element="step", stat = 'density', common_norm=False, fill = True , kde = True, kde_kws={'bw_adjust': 2})
        plt.xlabel('alpha')
        plt.title('Alpha - ' + str(lag_t) + ' sec')
        plt.axvline(x=D_alpha_[D_alpha_['Loc']=='inside'].alpha.median(), color='salmon', linestyle='--')
        plt.axvline(x=D_alpha_[D_alpha_['Loc']=='outside'].alpha.median(), color='purple', linestyle='--')
        plt.tight_layout()
        #plt.savefig(folder_path + '/alpha_' + str(lag_t) +  'sec_power_law.pdf', dpi = 300)
        plt.show()




In [ ]:
#resolution calculation

plt.rc('font', family='Helvetica', size=20)
plt.figure(figsize=(8, 6))

def row_single_differences(group):
    # Compute differences within each group
    group['X_diff'] = group['X'].diff(periods=1)
    group['Y_diff'] = group['Y'].diff(periods=1)
    return group

mind_diff = min_total_total.groupby(['filename','track_id']).apply(row_single_differences, include_groups=False).reset_index()


plt.hist([mind_diff.X_diff*1000, mind_diff.Y_diff*1000], color=['chocolate','skyblue'], alpha=0.9, bins = 50)
plt.legend(['ΔX','ΔY'])
plt.title('st. dev. (ΔX): ' + str(round(np.std(mind_diff.X_diff)*1000,2)) + 
          'nm' + '\nst. dev. (ΔY): ' + str(round(np.std(mind_diff.Y_diff)*1000,2)) + 'nm') 
plt.xlim(-50.1,50.1)
plt.xlabel('difference between consecutive localizations (nm)')
plt.tight_layout()
sns.despine()



plt.savefig(directory_name + folder_name + '/'+ 'resolution.pdf', dpi = 300)
plt.show()

In [ ]:
#nanodomain density calculation

df = min_total_total
area_list = []


time_segment = 0.1
for (sample, filename, t, track_id), group in tqdm(df.groupby(['sample', 'filename', 'type', 'track_id'])):


    # Segmenting parameters
    window = int(time_segment / min_total_total.time_int.median())  # 100 ms window
    segment = int(len(group) / window)

        

    # Per-segment KDE + area calculation


    for i in range(segment):
        segment_data = group[i * window: (i + 1) * window]

        x = segment_data["X"].values
        y = segment_data["Y"].values

        # KDE grid boundaries
        xmin, xmax = group.X.min() - 0.2, group.X.max() + 0.2
        ymin, ymax = group.Y.min() - 0.2, group.Y.max() + 0.2

        xx, yy = np.meshgrid(np.linspace(xmin, xmax, int((xmax-xmin)*60)), np.linspace(ymin, ymax, int((ymax-ymin)*60)))
        dx = (xmax - xmin) / int((xmax-xmin)*60)
        dy = (ymax - ymin) / int((ymax-ymin)*60)

        positions = np.vstack([xx.ravel(), yy.ravel()])
        values = np.vstack([x, y])

        #kde = gaussian_kde(values)
        kde = gaussian_kde(values, bw_method='scott')
        density = kde(positions).reshape( int((ymax-ymin)*60),int((xmax-xmin)*60))


        max_density = density.max()
        total_density = np.sum(density) * dx * dy
        mean_density = np.mean(density)
   


        area_list.append({
            'sample':sample,
            'filename': filename,
            'type':t,
            'track_id': track_id,
            'Loc': 'inside' if (segment_data['Loc'] == 'inside').sum() > (segment_data['Loc'] == 'outside').sum() else 'outside',
            'segment': i,
            'max_density': max_density,
            'total_density': total_density,
            'mean_density': mean_density,
                
        })
            
dense_df = pd.DataFrame(area_list)



threshold = dense_df.max_density.median()
area_list = []



folder_path = os.path.join(directory_name, 'density_analysis_' + str(threshold))
os.makedirs(folder_path, exist_ok=True)

for time_w in np.arange(0.1, 0.11, 0.1):
    
    df['InDenseRegion'] = False
    
    time_segment = time_w
    for (sample, filename, t, track_id), group in tqdm(df.groupby(['sample', 'filename', 'type', 'track_id'])):


        folder_path_sample = os.path.join(folder_path, sample)
        os.makedirs(folder_path_sample, exist_ok=True)

        folder_name = filename.rsplit(' ', 1)[0]
        #folder_path2 = os.path.join(folder_path_sample, folder_name + '_' + str(time_w))
        folder_path2 = os.path.join(folder_path_sample, folder_name + '_' + str(time_w) + '_' + str(threshold))
        os.makedirs(folder_path2, exist_ok=True)

            # Segmenting parameters
        window = int(time_segment / min_total_total.time_int.median())  # 100 ms window
        segment = int(len(group) / window)

        #print(f"Total segments: {segment}")
        #print(f"Total points in track: {len(group)}")

        param = pd.read_csv(directory_name + folder_name + '/' + filename  + '/params.csv')

        def string_to_float_array(string):
            return [float(x) for x in string.split(',')]
        L_str = param['0'][0:1][0]
        L = string_to_float_array(L_str[L_str.find('[')+1 : L_str.find(']')])
        O_str = "".join(param['0'][1:2][1])
        O = string_to_float_array(O_str[O_str.find('[')+1 : O_str.find(']')])
        Lengths = [i * 1e6 for i in L]
        Offsets = [i * 1e6 for i in O]
        cmap3 = LinearSegmentedColormap.from_list('mycmap', ['white', 'darkgreen'])


        min_LDs = directory_name + folder_name + '/' + filename  + '/LDs.tif'
        IMAGE = io.imread(min_LDs)
        pre = IMAGE[0]
        post = IMAGE[1]
        MIP= np.max(IMAGE, axis=0)
        image_minmax_scaled = exposure.rescale_intensity(MIP)
        percentiles = np.percentile(image_minmax_scaled, (75, 99.99))
        scaled = exposure.rescale_intensity(MIP,in_range=tuple(percentiles))


        # Full track overview
        plt.figure(figsize=(7, 7))
        plt.imshow(scaled, cmap='Greens', extent=[0, Lengths[0], Lengths[1], 0])
        sns.kdeplot(data=group, x="X", y="Y", levels=15, thresh=0.3, alpha=0.8, color='salmon', fill=True)
        sns.lineplot(data=group, x="X", y="Y", color='k', linewidth=0.05, sort=False, estimator=None)

        plt.axis('scaled')
        plt.xlim(group.X.min() - 0.2, group.X.max() + 0.2)
        plt.ylim(group.Y.min() - 0.2, group.Y.max() + 0.2)
        plt.title(f"{filename}, Track {track_id}" + '\ntrack_length: ' + str(round(group.time_int.sum(), 2)) + ' sec')
        plt.gca().invert_yaxis()
        plt.savefig(folder_path_sample + '/'+ f"{filename}, Track {track_id}" + '\ntrack_length: ' + str(round(group.time_int.sum(), 2)) + '_sec.pdf' )
        #plt.show()
        plt.clf()

            # Per-segment KDE + area calculation


        for i in range(segment):
            segment_data = group[i * window: (i + 1) * window]

            x = segment_data["X"].values
            y = segment_data["Y"].values

            # KDE grid boundaries
            xmin, xmax = group.X.min() - 0.2, group.X.max() + 0.2
            ymin, ymax = group.Y.min() - 0.2, group.Y.max() + 0.2

            xx, yy = np.meshgrid(np.linspace(xmin, xmax, int((xmax-xmin)*60)), np.linspace(ymin, ymax, int((ymax-ymin)*60)))


            positions = np.vstack([xx.ravel(), yy.ravel()])
            values = np.vstack([x, y])

            #kde = gaussian_kde(values)
            kde = gaussian_kde(values, bw_method='scott')
            density = kde(positions).reshape( int((ymax-ymin)*60),int((xmax-xmin)*60))

            # Calculate area above threshold
            threshold = threshold
            mask1 = density > threshold
            dx = (xmax - xmin) / int((xmax-xmin)*60)
            dy = (ymax - ymin) / int((ymax-ymin)*60)
            area_um2_dense = np.sum(mask1) * dx * dy

            mask2 = density > 2
            area_um2_total = np.sum(mask2) * dx * dy


            # Evaluate KDE at each localization point
            local_density_vals = kde(np.vstack([x, y]))  # 1D array, same length as number of points
            
            
            # Determine which localizations fall into dense region
            dense_mask = local_density_vals > threshold
            
            # Count transitions into and out of dense region
            dense_mask_series = pd.Series(dense_mask).astype(int)
            transitions = dense_mask_series.diff().fillna(0)

            # Count entries (0 -> 1) and exits (1 -> 0)
            entries = (transitions == 1).sum()
            exits = (transitions == -1).sum()
            
            
            # Store this in original DataFrame by matching index
            segment_indices = segment_data.index
            df.loc[segment_indices, 'InDenseRegion'] = dense_mask
            
            # Dwell time calculation inside dense region
            segment_time_ints = segment_data['time_int'].values
            dwell_time_inside = np.sum(segment_time_ints[dense_mask])
            
            # Count how many localizations fall in high-density zones
            num_spots_dense = np.sum(local_density_vals > threshold)
            total_spots = len(local_density_vals)
            fraction_dense = num_spots_dense / total_spots if total_spots > 0 else np.nan

            max_density = density.max()
            total_density = np.sum(density) * dx * dy
            mean_density = np.mean(density)
            total_density_at_spots = np.sum(local_density_vals)
            sum_density = np.sum(density)
            
            gap_tolerance = 3               # max allowed non-dense points inside a dense stretch
            min_dwell_points = 3            # minimum number of dense localizations to count as dwell
            segment_time_ints = segment_data['time_int'].values
            
            if dense_mask.sum() < min_dwell_points:
                # Not enough dense points to ever form a valid episode
                dwell_episodes = []
                num_episodes = 0
                mean_dwell = 0
                total_dwell_time = 0
            else:
                

                # Initialize
                dwell_episodes = []
                current_dwell_time = 0.0
                current_dwell_points = 0
                gap_count = 0
                in_dwell = False

                for is_dense, dt in zip(dense_mask, segment_time_ints):
                    if is_dense:
                        current_dwell_time += dt
                        current_dwell_points += 1
                        gap_count = 0
                        in_dwell = True
                    else:
                        if in_dwell and gap_count < gap_tolerance:
                            current_dwell_time += dt
                            current_dwell_points += 1
                            gap_count += 1
                        else:
                            # Finalize dwell episode if it meets minimum length
                            if current_dwell_points >= min_dwell_points:
                                dwell_episodes.append(current_dwell_time)
                            # Reset everything
                            current_dwell_time = 0.0
                            current_dwell_points = 0
                            gap_count = 0
                            in_dwell = False

                # Catch any trailing dwell
                if current_dwell_points >= min_dwell_points:
                    dwell_episodes.append(current_dwell_time)

                # Final results
                num_episodes = len(dwell_episodes)
                mean_dwell = np.mean(dwell_episodes) if num_episodes > 0 else 0
                total_dwell_time = np.sum(dwell_episodes)

            

            area_list.append({
                'sample':sample,
                'time_window':time_w,
                'filename': filename,
                'type':t,
                'track_id': track_id,
                'Loc': 'inside' if (segment_data['Loc'] == 'inside').sum() > (segment_data['Loc'] == 'outside').sum() else 'outside',
                'segment': i,
                'max_density': max_density,
                'area_um2_dense': area_um2_dense,
                'area_um2_total': area_um2_total,
                'ratio': area_um2_dense/area_um2_total,
                'total_density': total_density,
                'mean_density': mean_density,
                'sum_density' : sum_density,
                'total_density_at_spots': total_density_at_spots,
                'num_spots_dense': num_spots_dense,
                'total_spots': total_spots,
                'ratio_spots': fraction_dense,
                'spot_per_dense':num_spots_dense/area_um2_dense,
                'spot_per_total': total_spots/area_um2_total,
                'area_spot_ratio': (num_spots_dense/area_um2_dense)/(total_spots/area_um2_total),
                'num_dwell_episodes': num_episodes,
                'mean_dwell_time': mean_dwell,
                'dwell_time_dense_sec': total_dwell_time,
                'in_out_transitions': entries + exits,
                'entries_to_dense': entries,
                'exits_from_dense': exits,
                'threshold': threshold
            })

            # Plot
            #plt.figure(figsize=(7, 7))
            sns.lineplot(data=group, x="X", y="Y", color='grey', alpha=0.5, linewidth=0.2, sort=False, estimator=None)
            sns.lineplot(data=segment_data, x="X", y="Y", linewidth=0.4, color='k', sort=False, estimator=None, alpha=0.8)


            levels = np.linspace(threshold, 180, 10)
            
            plt.contourf(xx, yy, density, levels=levels, cmap='Reds', alpha=1)
            plt.contour(xx, yy, density, levels=[threshold], colors='blue', linewidths=1.5)
            plt.contour(xx, yy, density, levels=[2], colors='magenta', linewidths=0.5)

            time_ms = round(group[0:(i + 1) * window].time_int.sum() * 1000, 1)
            #plt.title(f"{time_ms} ms — Area > {threshold}: {area_um2_dense:.4f} µm²")
            plt.title(f"{filename}, Track {track_id}, {time_ms} ms\nArea > {threshold}: {area_um2_dense:.4f} µm²")
            plt.axis('scaled')
            plt.xlim(xmin, xmax)
            plt.ylim(ymin, ymax)
            plt.gca().invert_yaxis()
            plt.xlabel("X (µm)")
            plt.ylabel("Y (µm)")
            plt.tight_layout()
            plt.savefig(folder_path2 + '/'+ f"{filename}, Track {track_id}, {time_ms} ms\nArea > {threshold}: {area_um2_dense:.4f} µm².pdf")
            #plt.show()
            plt.clf()
    
area_df_WT = pd.DataFrame(area_list)
area_df_WT.to_csv(folder_path + '/density_areas_100ms.csv')

In [ ]:
#sliding window msd_and D - 5ms windows based on nanodomain behavior

time_frame = 0.005

def msd_analysis(df_sliding, window_size=int(time_frame/(df.time_int.median()))):
    results = []

    for (sample, filename, track_id), track in tqdm(df_sliding.groupby(['sample','filename', 'track_id'])):
        track = track.sort_values(by='time').reset_index(drop=True)
        num_points = len(track)

        if num_points < window_size:
            continue

        for i in range(num_points - window_size + 1):
            window = track.iloc[i:i+window_size]
            time_lags = window['time'].values - window['time'].values[0]
            displacements = (window['X'].values - window['X'].values[0])**2 + \
                            (window['Y'].values - window['Y'].values[0])**2

            msd_values = [np.mean(displacements[:j+1]) for j in range(1, window_size)]
            time_lags = time_lags[1:]

            if len(msd_values) < 2:
                continue

            D_values = [msd / (4 * t) if t > 0 else np.nan for msd, t in zip(msd_values, time_lags)]
            D_avg = np.nanmean(D_values) if len(D_values) > 0 else np.nan

            # Determine dense region correlation
            n_true = window['InDenseRegion'].sum()
            ratio_dense = n_true / window_size

            if ratio_dense >= 0.75:
                correlation = 'Dense'
            elif ratio_dense <= 0.25:
                correlation = 'not_Dense'
            else:
                correlation = 'intermediate'

            results.append([
                sample, filename, track_id, window['time'].values[0], D_avg, correlation
            ])

    rolling_D = pd.DataFrame(results, columns=['sample','filename', 'track_id', 'start_time', 'D', 'correlation'])
    return rolling_D


df_sliding = df#[min_total_total['type']=='single']
rolling_D = msd_analysis(df_sliding)

# Merge back other metadata if needed (e.g., Loc, X, Y)
rolling_D = rolling_D.merge(
    df[['sample','filename', 'track_id', 'Loc', 'time', 'type', 'X', 'Y']],
    left_on=['sample','filename', 'track_id', 'start_time'],
    right_on=['sample','filename', 'track_id', 'time'],
    how='left'
)

rolling_D.drop(columns='time', inplace=True)

df.to_csv(folder_path +  '/tracks_full_data.csv')
rolling_D.to_csv(folder_path +  '/sliding_msd_' + str(time_frame*1000) + 'ms_segments.csv')

plt.rc('font', family='Helvetica', size=20)
plt.figure(figsize=(6, 6))




In [ ]:
#repetetive nanodomain engagement - how many times each dense region is captured

area_df_1['dense_state'] = (area_df_1['ratio_spots'] > 0.7) 

def count_transitions(states):
    arr = states.values  # convert to numpy
    return np.sum(arr[:-1] != arr[1:])

transition_df = (
    area_df_WT
    .sort_values(['time_window','sample','type','filename', 'track_id', 'segment'])
    .groupby(['time_window','sample','type', 'filename', 'track_id', 'Loc'])['dense_state']
    .apply(count_transitions)
    .reset_index(name='num_transitions')
)



data = transition_df[(transition_df['type']=='single')  ]

sns.boxplot(data = data, x = 'sample', y = 'num_transitions', hue = 'Loc', 
             palette = 'magma', log_scale = False, hue_order = ['outside','inside'], order = ['GPAT4', 'HSD17B13'],
           fliersize = 1)
plt.ylabel('number of transitions between states')

plt.title(data.groupby(['sample', 'Loc']).num_transitions.mean())
plt.ylim(-0.5,10.5)
sns.despine()
stat_data = data[data['sample']=='HSD17B13']
mann_whit = stats.ttest_ind(stat_data[stat_data['Loc'] == 'outside'].num_transitions, 
                               stat_data[stat_data['Loc'] == 'inside'].num_transitions, nan_policy='omit')
print(mann_whit)


#plt.savefig(folder_path + '/transitions_between_dense_segments.pdf')
plt.show()

In [ ]:
#Mean vs Total density at nanodomains

density_df = data.rename(columns={
    'mean_kde_at_spots': 'mean_density',
    'total_kde_in_segment': 'total_density'
})


plt.figure(figsize=(7,5))
sns.scatterplot(
    data=density_df[density_df['sample']=='HSD17B13'],
    x='total_density',
    y='mean_density',
    hue='Loc',  # or color by 'inside' vs 'outside'
    alpha=0.6,
    edgecolor=None,
    palette = 'magma',
    hue_order = ['outside','inside']
)
plt.title('Mean vs. Total KDE Density')
plt.xlabel('Total KDE Density (explored area)')
plt.ylabel('Mean KDE Density at Spots (visited regions)')
plt.grid(True)
plt.tight_layout()
#plt.savefig(folder_path + '/mean_vs_total_density_HSD17B13.pdf')


sns.lmplot(
    data=density_df[density_df['sample']=='HSD17B13'],
    x='total_density',
    y='mean_density',
    hue='Loc',
    aspect=1.3,
    height=5,
    palette = 'magma',
    hue_order = ['outside','inside'],
    scatter_kws={'alpha':0.4},

)
#plt.savefig(folder_path + '/fit_mean_vs_total_density_HSD17B13.pdf')
plt.show()

In [ ]:
#CDF for density enrichment at nanodomains

var = 'density_efficiency'
density_df['density_efficiency'] = density_df['mean_density'] / density_df['sum_density']


data = density_df[(density_df['sample']=='HSD17B13') ]

sns.histplot(data = data,  x = var, hue = 'Loc', bins = 60,
                 element="step", stat = 'density', palette = 'magma_r', common_norm=False, 
                 fill = True, log_scale = True, kde = False, hue_order = ['inside', 'outside'], cumulative = True )

stat_data = data

mann_whit = stats.mannwhitneyu(stat_data[stat_data['Loc'] == 'outside'][var], stat_data[stat_data['Loc'] == 'inside'][var], nan_policy='omit')
ttest = stats.ttest_ind(stat_data[stat_data['Loc'] == 'outside'][var], stat_data[stat_data['Loc'] == 'inside'][var], nan_policy='omit')

print(mann_whit)
print(ttest)


plt.xlabel('density enrichment')
plt.ylabel('CDF Probability')

plt.title(density_df.groupby(['sample', 'Loc'])[var].median())

#plt.savefig(folder_path + '/density_enrichment_CDF_GPAT4.pdf')
plt.show()